## 4. Track A — one directed 3-cycle (Paper 033 `play_demo`)

The opening cell used a **chorded** triangle $A,B,C$ with a chord $A \to C$ and claims $(1,2,0)$ vs $(1,2,3)$. That cell stays. This section is a **different** graph — the Course I on-ramp example:

- vertices $a,b,c$
- edges $a \to b$ (weight 1), $b \to c$ (weight 2), $c \to a$ (weight 2)
- claims $c = (1,1,1)$
- host $\Phi_1 = 3$  **[Pinned]** `compute_phi1` / `play_demo.py`

Short video of this single example: `notebooks/scenes/three_cycle_phi1.py` (matplotlib+ffmpeg; optional Manim class `ThreeCyclePhi1`).

**Predict-then-run.** Before executing the next cell, write `CONSISTENT` / `CONTRADICTION` and $\Phi_1$. The checked-in values are the worked solution; change a claim without updating them and the cell fails.


In [ ]:
import importlib.util
import sys
from fractions import Fraction
from pathlib import Path


def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def _repo_root() -> Path:
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path(".").resolve()
    cur = here
    while cur.parent != cur and not (cur / "education").is_dir():
        cur = cur.parent
    return cur


def _frontend_dir() -> Path:
    return _repo_root() / "education" / "education_app" / "frontend"


_NB_DIR = _frontend_dir()
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from notebook_lib import GovernorAdapter

_paper = (
    _repo_root()
    / "atlas"
    / "coda_atlas"
    / "F004_papers"
    / "015_pfp"
    / "005_pfp_practitioners_field_manual"
    / "code"
)
_governor = _load("l1_symbol_game_governor", _paper / "l1_symbol_game_governor.py")
Cochain1 = _governor.Cochain1
ConstraintGraph = _governor.ConstraintGraph
Edge = _governor.Edge
compute_phi1 = _governor.compute_phi1

STUDENT_PREDICTION = {"verdict": "CONTRADICTION", "phi1": Fraction(3)}

play_vertices = ("a", "b", "c")
play_edges = (("a", "b", 1), ("b", "c", 2), ("c", "a", 2))
play_claims = {("a", "b"): Fraction(1), ("b", "c"): Fraction(1), ("c", "a"): Fraction(1)}

graph = ConstraintGraph(
    vertices=play_vertices,
    edges=tuple(Edge(u, v, w) for u, v, w in play_edges),
)
phi1, gauge = compute_phi1(graph, Cochain1(play_claims))
adapter = GovernorAdapter.project_l1_exact(list(play_vertices), list(play_edges), play_claims)

assert isinstance(phi1, Fraction)
assert phi1 == Fraction(3)
assert adapter.primary_obstruction == phi1
assert STUDENT_PREDICTION["phi1"] == phi1
assert STUDENT_PREDICTION["verdict"] == ("CONSISTENT" if phi1 == 0 else "CONTRADICTION")
print(f"[Pinned] compute_phi1 play_demo cycle: Phi_1 = {phi1}, gauge = {gauge}")
print(f"[Pinned] GovernorAdapter agrees: Phi_1 = {adapter.primary_obstruction}")


In [ ]:
import importlib.util


def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


_paper = _repo_root() / "atlas" / "coda_atlas" / "F004_papers" / "015_pfp" / "005_pfp_practitioners_field_manual" / "code"
_load("l1_symbol_game_governor", _paper / "l1_symbol_game_governor.py")
run_session = _load("play_demo", _paper / "play_demo.py").run_session

session = run_session(verbose=True)
assert session["phi_after_assert"] == phi1 == Fraction(3)
assert session["bad_status"] == "DefectIncrease"
assert session["bad_admitted"] is False
assert session["repair_status"] == "ExactRepair"
assert session["repair_before"] == Fraction(3)
assert session["repair_after"] == Fraction(0)
assert session["verify_admitted"] is True
assert session["phi_after_retract"] == Fraction(3)
print("[Pinned] play_demo fail-closed session matches compute_phi1")

In [ ]:
# One-parameter transfer: change exactly one claim, c(c -> a) = 1 -> -2.
# Cycle sum 1+1-2 = 0. Host class must flip CONTRADICTION -> CONSISTENT.
TRANSFER_PREDICTION = {"verdict": "CONSISTENT", "phi1": Fraction(0)}

transfer_claims = dict(play_claims)
transfer_claims[("c", "a")] = Fraction(-2)
t_phi, t_gauge = compute_phi1(graph, Cochain1(transfer_claims))
assert t_phi == Fraction(0)
assert TRANSFER_PREDICTION["phi1"] == t_phi
assert TRANSFER_PREDICTION["verdict"] == ("CONSISTENT" if t_phi == 0 else "CONTRADICTION")
print(f"[Pinned] transfer c(c->a)=-2: Phi_1 = {t_phi}, gauge = {t_gauge}, CONSISTENT")

## 5. Scope: CAN-Show vs. CANNOT-Show

> **CAN-Show**:
> - Demonstrates that simplicial coboundary operators detect cycle inconsistencies with zero floating-point loss.
> - Demonstrates that $L_1$ projection is idempotent ($P_S^2 = P_S$) on finite graphs.
> - Replays Paper 033 `play_demo.py` on the directed 3-cycle: $\Phi_1 = 3$, a defect-increasing CUT is refused (`DefectIncrease`, `admitted=False`), ExactRepair lands at 0, RETRACT restores 3. **[Pinned]**
> - One-parameter transfer: changing only $c(c \to a)$ from $1$ to $-2$ flips the host class to CONSISTENT with $\Phi_1 = 0$. **[Pinned]**
>
> **CANNOT-Show**:
> - Does not show that every frustrated cycle can be resolved without retracting at least one claim.
> - Does not solve NP-hard combinatorial optimization problems in polynomial time.
> - The 3-cycle is one finite weighted graph. It is not a claim that language models are cochain complexes.


## 6. Student Exercises

### Exercise 05.1: Cycle Frustration
This is the Track A directed cycle (not the chorded opening cell). Edges $a \to b, b \to c, c \to a$, claims $(1,1,1)$. What is the cycle sum, and what does the host return?

<details>
<summary><b>Click to reveal solution</b></summary>
The cycle sum is $1 + 1 + 1 = 3 \neq 0$. Coboundaries have cycle sum 0, so this is not in $B^1$. `compute_phi1` returns $\Phi_1 = 3$ and the host verdict is CONTRADICTION. **[Pinned]**
</details>

### Exercise 05.2: One-parameter transfer
Change exactly one claim so the host class flips to CONSISTENT. Check $\Phi_1$ with `compute_phi1`, not by averaging.

<details>
<summary><b>Click to reveal solution</b></summary>
$c(c \to a) = -2$ gives cycle sum $0$ and $\Phi_1 = 0$ (gauge $f(a)=0, f(b)=1, f(c)=2$). Any other single-claim change that leaves the cycle sum nonzero stays CONTRADICTION.
</details>


## 7. Anchors

- **Paper 032 Sections**: §3.1–§3.3
- **Paper 033**: §5 worked session (`play_demo.py`)
- **Ledger IDs**: `PO032-C29` through `PO032-C36`
- **Code**: `papers/032_the_symbol_game_three_layer_sovereignty_and_discrete_l1_projection/code/l1_symbol_game_governor.py` (`compute_phi1`, module-level)
- **Code**: `papers/033_the_symbol_game_practitioners_field_manual/code/play_demo.py`
- **Video**: `notebooks/scenes/three_cycle_phi1.py` (Track A video 3)


> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `05_discrete_l1_coboundary_projection`
- The code checks one finite example. It is not a general proof.
